# Trabajo Práctico — Entrega 2

**Alumno:** Patricio Gerpe
**Materia:** Data Mining — Esp. en Explotación de Datos y Descubrimiento de Conocimiento (UBA Exactas)

Esta entrega continúa el trabajo de la Entrega 1 incorporando las técnicas vistas en las clases 2 a 5: tratamiento de **valores atípicos**, **imputación de datos faltantes** y **creación de nuevos atributos**. Se mantienen los filtros de la entrega anterior (sólo `venta`, `dolares`, CABA / Buenos Aires y tipos de propiedad presentes en `a_predecir.csv`).

Estructura del notebook:

1. Lectura de datos (`entrenamiento.db` + `a_predecir.csv`).
2. Análisis exploratorio (recap entrega 1).
3. Limpieza y transformación:
   - 2.1 Filtrado.
   - 2.2 Tratamiento de valores atípicos (IQR, Z-score, IsolationForest).
   - 2.3 Imputación de faltantes (mediana / moda / KNN + indicador de ausencia).
   - 2.4 Creación de nuevos atributos (parseo de la columna `features`, barrio, etc.).
4. Entrenamiento del modelo (RandomForestRegressor — sección no modificable).
5. Predicción sobre `a_predecir.csv` y override por *Hot Deck* de duplicados conocidos.

In [ ]:
import re
import sqlite3
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy.stats import zscore

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)

In [ ]:
# En Colab montamos Drive; en local (donde se valida antes de subir) lo salteamos.
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Modo local: se asume que los datasets están en ./datasets")

In [ ]:
# --- Parámetros del experimento ---
# Estos valores se pueden override desde el orquestador (entregas/run_entrega.py)
# vía variables de entorno. Cuando se ejecuta el notebook a mano, se usan los
# defaults definidos acá.
import os, json, hashlib, datetime, platform

ENTREGA = os.environ.get("EXPERIMENT_ENTREGA", "entrega_2")
NOMBRE  = os.environ.get("EXPERIMENT_NAME",    "v1")
DESCRIPCION = os.environ.get(
    "EXPERIMENT_DESC",
    "Filtro CABA ampliado + parseo features + outliers IQR/IF + imputación mediana/moda + Hot Deck por description",
)

# Diccionario de log que se va llenando a lo largo del notebook y se serializa al final.
EXPERIMENT_LOG = {
    "entrega": ENTREGA,
    "nombre": NOMBRE,
    "descripcion": DESCRIPCION,
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "hostname": platform.node(),
    "python": platform.python_version(),
    "in_colab": IN_COLAB,
    "params": {},
    "data": {},
    "metricas": {},
    "kaggle": {
        "rmse": None,
        "submitted_at": None,
        "leaderboard_position": None,
    },
}
print(f"Experimento: {ENTREGA}/{NOMBRE}")
print(f"Descripción: {DESCRIPCION}")

## 0. Lectura de datos

Levantamos `entrenamiento.db` (train, ~1.29 M filas) y `a_predecir.csv` (test, 13 471 filas). Se cargan ambos juntos para poder aplicar el mismo preprocesamiento — en particular, la **detección de duplicados** entre train y test, que se usa luego como imputación tipo *Hot Deck* del precio.

In [ ]:
DIR = "/content/drive/MyDrive/datos/propiedades" if IN_COLAB else "datasets"

In [ ]:
engine = sqlite3.connect(f"{DIR}/entrenamiento.db")
df_ent = pd.read_sql("SELECT * FROM entrenamiento", engine, index_col="id")
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")

df_ent["__src__"] = "train"
df_ap["__src__"] = "test"

print(f"train: {df_ent.shape}  |  test: {df_ap.shape}")

In [ ]:
df_ent.shape, df_ap.shape

In [ ]:
df_ent.columns.tolist()

## 1. Entender los datos (AID)

Recap del análisis exploratorio de la Entrega 1:

* `a_predecir` es **únicamente CABA** (`location_1 ∈ {Capital Federal, Ciudad Autónoma de Buenos Aires}`), **siempre `venta`** y prácticamente todo en **dólares** → ese es el universo a modelar.
* `train` cubre todo el país, varias operaciones (venta / alquiler / temporal) y dos monedas, además de un fuerte ruido en `price` (mezcla de moneda y outliers extremos).
* `property_type` tiene categorías redundantes (`departamento` / `departamentos`, `casa` / `casas`).
* Hay muchos faltantes en geolocalización (`lat`, `lon`), `publication_date`, `publisher_id` y en la propia variable objetivo.

In [ ]:
nulos = pd.concat(
    [
        df_ent.isna().mean().rename("train_%"),
        df_ap.isna().mean().rename("test_%"),
    ],
    axis=1,
).sort_values("train_%", ascending=False).round(3)
nulos

## 2. Limpiar y transformar los datos (DM)

## 2.1. Filtrado de datos

Mantenemos los filtros decididos en la **Entrega 1**, refinados con un **EDA train-vs-test** (script `eda_train_vs_test.py`):

* `operation_type == "venta"` — cubre **100 %** del test.
* `currency_type == "dolares"` — **99.9 %** (8 filas en pesos, despreciable).
* `property_type ∈ {departamento, departamentos, casa, casas, ph, cochera}` — cubre **100 %** del test, con la unificación de variantes con/sin "s".
* `price ∈ [USD 5 000, USD 3 000 000]` — el corte sólo deja afuera el **0.3 %** de las filas de train (108 con `< 5K` que son ruido de carga y 228 con `> 3M` que son mansiones extremas). El P99.9 del precio post-filtro es 2.5 M USD ⇒ el corte superior cubre prácticamente todo.
* `location_1 ∈ CABA` — **mejora respecto a Entrega 1**. El filtro original `{Capital Federal, Ciudad Autónoma de Buenos Aires}` perdía publicaciones de CABA mal etiquetadas como `location_1='Buenos Aires'`. Las recuperamos cuando además su `location_2` o `location_3` coincide con un barrio observado en el test → **+3 341 filas (+2.8 %)** de entrenamiento legítimo.

El filtro **se aplica sólo al train** (sobre el test no se filtra: hay que predecir todas las filas).


In [ ]:
CABA_L1 = {"Capital Federal", "Ciudad Autónoma de Buenos Aires"}
PROP_OK = {"departamento", "departamentos", "casa", "casas", "ph", "cochera"}

# Universo geográfico: CABA estricto en location_1, MÁS las filas con
# location_1='Buenos Aires' cuyo location_2 o location_3 aparece como barrio CABA
# en el test (recupera ~3.3K filas mal etiquetadas — ver eda_train_vs_test.py).
caba_loc2 = set(df_ap["location_2"].dropna().unique())
caba_loc3 = set(df_ap["location_3"].dropna().unique())

mask_caba = (
    df_ent["location_1"].isin(CABA_L1)
    | (
        df_ent["location_1"].eq("Buenos Aires")
        & (df_ent["location_2"].isin(caba_loc2)
           | df_ent["location_3"].isin(caba_loc3))
    )
)

mask = (
    df_ent["operation_type"].eq("venta")
    & df_ent["currency_type"].eq("dolares")
    & mask_caba
    & df_ent["property_type"].isin(PROP_OK)
    & df_ent["price"].notna()
    & df_ent["price"].between(5_000, 3_000_000)
)
df_ent = df_ent.loc[mask].copy()

# Unificación de categorías redundantes (vista en Entrega 1)
df_ent["property_type"] = df_ent["property_type"].replace(
    {"departamentos": "departamento", "casas": "casa"}
)
df_ap["property_type"] = df_ap["property_type"].replace(
    {"departamentos": "departamento", "casas": "casa"}
)

print(f"train luego de filtrar: {df_ent.shape}")
print(df_ent["property_type"].value_counts())

EXPERIMENT_LOG["params"]["filtros"] = {
    "caba_l1_estricto": sorted(CABA_L1),
    "caba_l1_ampliado_via_loc2_loc3": True,
    "property_type": sorted(PROP_OK),
    "operation_type": "venta",
    "currency_type": "dolares",
    "price_min": 5000,
    "price_max": 3_000_000,
}
EXPERIMENT_LOG["data"]["train_post_filtro"] = list(df_ent.shape)

In [ ]:
## --- 2.4 Creación de nuevos atributos (se hace ANTES de outliers/imputación
## --- porque las nuevas variables — m², dormitorios, baños — son justamente
## --- las que vamos a usar para detectar outliers e imputar.) ---

def parse_features(serie: pd.Series) -> pd.DataFrame:
    """De la cadena 'features' (semicolons) extrae numéricos y flags binarios."""
    AMENITIES = [
        "balcon", "garage", "cochera", "pileta", "parrilla",
        "aire acondicionado", "calefaccion", "gas natural",
        "internet", "seguridad", "alarma", "gimnasio", "jardin",
        "cuarto de servicio", "cocina equipada", "bodega",
    ]
    s = serie.fillna("").astype(str).str.lower()
    # Normalizar acentos para que "calefacción" == "calefaccion", "baño" == "bano", etc.
    # Sin esto, tener "calefacción" y "calefaccion" en la lista produce la misma
    # columna dos veces y la segunda asignación sobreescribe la primera (pérdida silenciosa).
    for src, dst in [("á","a"),("é","e"),("í","i"),("ó","o"),("ú","u"),("ñ","n")]:
        s = s.str.replace(src, dst, regex=False)

    out = pd.DataFrame(index=serie.index)
    out["n_dormitorios"] = s.str.extract(r"(\d+)\s*dormitor", expand=False).astype(float)
    # Tras normalizar ñ→n, "baño" queda como "bano"
    out["n_banos"]       = s.str.extract(r"(\d+)\s*bano",     expand=False).astype(float)
    # m[²2] evita falsos positivos: "15 metros de frente;91 m²" extraía 15 en vez de 91
    out["m2"]            = s.str.extract(r"(\d+)\s*m[²2]",    expand=False).astype(float)
    for a in AMENITIES:
        out[f"f_{a.replace(' ', '_')}"] = s.str.contains(a, regex=False).astype(int)
    return out

# Atributos extra a partir de la dirección / locación
def extra_attrs(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["len_descripcion"] = df["description"].fillna("").str.len()
    out["n_features"]      = df["features"].fillna("").str.count(";")
    # `location_3` en CABA suele ser el barrio; lo dejamos como categórica.
    out["barrio"]          = df["location_3"].fillna("desconocido").astype(str)
    return out

feat_ent = pd.concat([parse_features(df_ent["features"]), extra_attrs(df_ent)], axis=1)
feat_ap  = pd.concat([parse_features(df_ap["features"]),  extra_attrs(df_ap)],  axis=1)

df_ent = pd.concat([df_ent, feat_ent], axis=1)
df_ap  = pd.concat([df_ap,  feat_ap],  axis=1)

print("Nuevas columnas:", feat_ent.columns.tolist())
df_ent[["n_dormitorios", "n_banos", "m2"]].describe()

In [ ]:
# -- EDA de las nuevas variables numéricas: base para definir los umbrales de outliers --
# Mostramos percentiles extremos para fundamentar los cortes que se aplicarán en 2.2.
print("=== Percentiles de m2, n_dormitorios, n_banos (antes de cualquier tratamiento) ===")
percs = [0.001, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
display(df_ent[["m2", "n_dormitorios", "n_banos"]].quantile(percs).round(1).T)

# La distribución de dormitorios y baños deja ver dónde están los errores de parseo
print("\n=== Frecuencia de n_dormitorios (valores raros son errores de parseo) ===")
print(df_ent["n_dormitorios"].value_counts(dropna=False).sort_index().head(25).to_string())
print("\n=== Frecuencia de n_banos ===")
print(df_ent["n_banos"].value_counts(dropna=False).sort_index().head(25).to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df_ent["m2"].clip(upper=600).hist(bins=80, ax=axes[0]); axes[0].set_title("m2 (clipped >600, raw)")
df_ent["n_dormitorios"].hist(bins=20, ax=axes[1]); axes[1].set_title("n_dormitorios (raw)")
df_ent["n_banos"].hist(bins=20, ax=axes[2]); axes[2].set_title("n_banos (raw)")
plt.tight_layout(); plt.show()

In [ ]:
# --- v5: features de calidad / estado y temporales desde description + features ---
# Hipótesis: floor (piso), a_estrenar, reciclado, suite, subte y pub_year/pub_month
# capturan señal de calidad y ciclo de mercado sin depender de agregados del train
# (evitamos el patrón de leakage confirmado en v2/v3).

def extract_quality_features(df_):
    """Extrae indicadores de calidad/estado y cercanía desde description + features."""
    import numpy as np
    out = {}
    desc = df_["description"].fillna("").astype(str).str.lower()
    feat = df_["features"].fillna("").astype(str).str.lower()
    for src, dst in [("á","a"),("é","e"),("í","i"),("ó","o"),("ú","u"),("ñ","n")]:
        desc = desc.str.replace(src, dst, regex=False)
        feat = feat.str.replace(src, dst, regex=False)
    combined = desc + " " + feat

    # Número de piso: "piso 8", "8° piso", "8 piso", "8º piso"
    # En CABA pisos altos → vista/precio premium
    _fp1 = combined.str.extract(r"piso\s+(\d{1,2})", expand=False)
    _fp2 = combined.str.extract(r"(\d{1,2})\s*(?:er|do|ro|to|[°º])\s*piso", expand=False)
    floor_vals = _fp1.fillna(_fp2).astype(float)
    # Capamos fuera de rango válido [1,50]
    floor_vals[~floor_vals.between(1, 50)] = float("nan")
    out["floor"] = floor_vals

    # Calidad / estado
    out["is_a_estrenar"]    = (desc.str.contains("a estrenar", regex=False) |
                               desc.str.contains("estrenar",   regex=False)).astype(int)
    out["is_reciclado"]     = desc.str.contains("reciclado",   regex=False).astype(int)
    out["has_suite"]        = combined.str.contains("suite",   regex=False).astype(int)
    out["has_service_room"] = (
        combined.str.contains("dependencia de servicio", regex=False) |
        combined.str.contains("cuarto de servicio",      regex=False)
    ).astype(int)

    # Accesibilidad transporte
    out["is_near_subway"] = (
        desc.str.contains("subte",    regex=False) |
        desc.str.contains("metrobus", regex=False)
    ).astype(int)

    return pd.DataFrame(out, index=df_.index)


def extract_temporal_features(df_):
    """Parsea publication_date (ej. '15 oct 2023') → pub_year, pub_month."""
    MESES = {"ene":1,"feb":2,"mar":3,"abr":4,"may":5,"jun":6,
             "jul":7,"ago":8,"sept":9,"sep":9,"oct":10,"nov":11,"dic":12}
    years, months = [], []
    pat = re.compile(r"(\d{1,2})\s+(\w+)\s+(\d{4})")
    for v in df_["publication_date"].fillna(""):
        m = pat.match(str(v).strip())
        if m:
            _, mes, a = m.groups()
            mes_n = MESES.get(mes.lower())
            years.append(int(a) if mes_n else float("nan"))
            months.append(mes_n if mes_n else float("nan"))
        else:
            years.append(float("nan"))
            months.append(float("nan"))
    return pd.DataFrame({"pub_year": years, "pub_month": months}, index=df_.index)


qual_ent  = extract_quality_features(df_ent)
qual_ap   = extract_quality_features(df_ap)
temp_ent  = extract_temporal_features(df_ent)
temp_ap   = extract_temporal_features(df_ap)

df_ent = pd.concat([df_ent, qual_ent, temp_ent], axis=1)
df_ap  = pd.concat([df_ap,  qual_ap,  temp_ap],  axis=1)

# Resumen rápido para verificar cobertura
print("Quality / temporal features — train:")
cols_show = ["floor","is_a_estrenar","is_reciclado","has_suite","pub_year","pub_month"]
print(df_ent[cols_show].describe().round(2))
print("\nCobertura pub_year train :", df_ent["pub_year"].notna().mean().round(3))
print("Cobertura floor train     :", df_ent["floor"].notna().mean().round(3))


## 2.2. Tratamiento de valores atípicos

Análisis univariado (IQR, Z-score) sobre las variables continuas más relevantes (`price`, `m2`) y multivariado con **IsolationForest** sobre el conjunto `(price, m2, n_dormitorios, n_banos)`.

* `price` ya quedó parcialmente acotada por el filtro [USD 5 K, USD 3 M], pero todavía contiene valores extremos dentro del rango.
* `m2` (parseado desde `features`) es la variable con **mayor cantidad de outliers**: hay publicaciones con 1 m² (claramente errores de carga / faltante disfrazado de cero) y otras con miles de m² (terrenos colados o errores tipográficos).
* La estrategia adecuada para `m2` es **acotar (winsorizar)** los valores fuera de rango razonable y dejar que la imputación posterior los trate como faltantes — porque eliminar la fila completa nos sacaría información útil de las demás columnas.
* Para *outliers multivariados* (combinaciones extrañas precio/superficie), usamos **IsolationForest** con contaminación moderada (1 %) y eliminamos esas filas del **train**. En el **test** no se elimina nada.

In [ ]:
# -- Tratamiento --
# 1) m2: winsorizar a NaN fuera de [10, 1500].
#    El EDA (percentiles anteriores) muestra que p0.1% está cerca de 10 m² y p99.9%
#    no supera los ~500 m² en CABA; valores fuera de [10, 1500] son errores de parseo
#    (e.g. "1 m²") o terrenos/countryhouses que no corresponden al universo del test.
#    Optamos por convertir a NaN (en vez de eliminar la fila) para conservar el resto
#    de columnas: el precio, barrio y tipo siguen siendo informativos aunque no tengamos m².
for df_ in (df_ent, df_ap):
    df_.loc[~df_["m2"].between(10, 1500), "m2"] = np.nan

# 2) n_dormitorios y n_banos: valores fuera de [0, 15] → NaN.
#    Los value_counts del EDA muestran que prácticamente la totalidad de los registros
#    tiene ≤ 10 dormitorios y ≤ 8 baños; valores > 15 son errores de parseo del texto
#    (e.g. "150 m2" capturado como dormitorios) y no representan propiedades reales en CABA.
for df_ in (df_ent, df_ap):
    df_.loc[~df_["n_dormitorios"].between(0, 15), "n_dormitorios"] = np.nan
    df_.loc[~df_["n_banos"].between(0, 15),       "n_banos"]       = np.nan

# 3) IsolationForest multivariado sólo en train.
#    Motivación: IQR y Z-score son univariados; no detectan combinaciones absurdas como
#    "5 m² a USD 500 000" o "300 m² a USD 20 000". IsolationForest (Clase 5) detecta
#    puntos aislados en el espacio multivariado usando árboles de aislamiento.
#    Parámetros:
#    - StandardScaler obligatorio: price (∼10⁵) y m2 (∼10²) difieren en 3 órdenes de
#      magnitud; sin escalar, el IF ignoraría las variables de menor varianza.
#    - contamination=0.01: el Z-score mostró ~1-2 % de outliers en price y m2 por separado;
#      fijamos 1 % para capturar sólo los casos más extremos en el espacio conjunto y
#      no descartar observaciones válidas en los bordes.
#    - Sólo filas con las 4 columnas completas (no tiene sentido aislar sobre NaN).
#    - Eliminamos del TRAIN (no del test: hay que predecir todas las filas).
iso_cols = ["price", "m2", "n_dormitorios", "n_banos"]
mask_complete = df_ent[iso_cols].notna().all(axis=1)
X_iso = df_ent.loc[mask_complete, iso_cols].astype(float)

scaler = StandardScaler()
X_iso_sc = scaler.fit_transform(X_iso)

iso = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
y_iso = iso.fit_predict(X_iso_sc)

ids_outliers = X_iso.index[y_iso == -1]
print(f"IsolationForest: {len(ids_outliers)} filas marcadas como outliers multivariados (de {mask_complete.sum()} completas).")

df_ent = df_ent.drop(index=ids_outliers)
print(f"train tras remover outliers multivariados: {df_ent.shape}")

EXPERIMENT_LOG["params"]["outlier_caps"] = {
    "m2": [10, 1500],
    "n_dormitorios": [0, 15],
    "n_banos": [0, 15],
}
EXPERIMENT_LOG["params"]["isolation_forest"] = {
    "contamination": 0.01,
    "cols": iso_cols,
    "scaled": "StandardScaler",
}
EXPERIMENT_LOG["data"]["outliers_if_eliminados"] = int(len(ids_outliers))
EXPERIMENT_LOG["data"]["train_post_outliers"] = list(df_ent.shape)

In [ ]:
# (celda intencionalmente vacía — el tratamiento de outliers se hace en la celda anterior)

## 2.3. Imputación de valores perdidos

Estrategia (vista en Clase 4 — Datos Faltantes):

* **Numéricas continuas** (`m2`, `lat`, `lon`, `len_descripcion`, `n_features`): imputación por **mediana** (`SimpleImputer(strategy='median')`). Es robusta a outliers que sobrevivan al paso anterior y conserva la escala. Probamos también `KNNImputer` pero la diferencia en el modelo final fue marginal y triplica el tiempo, así que para esta entrega nos quedamos con la mediana.
* **Numéricas discretas** (`n_dormitorios`, `n_banos`): imputación por **mediana** (entera) — la moda colapsaba todo a "2" y ocultaba la variabilidad.
* **Categóricas** (`property_type`, `barrio`): imputación por **moda** (`most_frequent`).
* **Flags binarios** parseados de `features` (`f_*`): no tienen NaN — si la palabra no estaba en el texto se asume que la propiedad no la tiene (= 0).
* **Marcador de ausencia**: agregamos `m2_was_na` para que el modelo sepa que la superficie fue imputada (técnica vista al final de la clase).

> **Sobre la pregunta B.3 ("¿qué pasa si una imputación sube el error?"):** generalmente significa que la suposición del método no se cumple — por ejemplo, asumir MAR cuando el faltante es MNAR (la ausencia depende del propio valor). En `lat`/`lon`, los nulos no son aleatorios: corresponden a publicaciones sin geolocalización válida que tienden a ser más viejas o de barrios menos cubiertos por properati; imputar por mediana acerca todo al centro geográfico y degrada el modelo. En esos casos conviene **no imputar y usar `MissingIndicator`** o imputar con valor "imposible" (e.g. 0) para que el árbol pueda partir por presencia/ausencia.

In [ ]:
# Marcador de ausencia para m2 (la columna con más outliers tratados como faltantes)
for df_ in (df_ent, df_ap):
    df_["m2_was_na"] = df_["m2"].isna().astype(int)

# --- v4: vuelta a la imputación global de v1 ---------------------------------
# Por qué: la CV exploratoria mostró que el RMSE del holdout único oscila
# ±2K solo por el seed (std=1997). v3 (mediana por barrio) "mejoraba" holdout
# en 1.8K (dentro del ruido) pero EMPEORÓ Kaggle de 93167 → 97423. Conclusión:
# H3 no transfiere bien al test público; volvemos a la estrategia de v1 que
# es la que mantiene el campeón Kaggle.
# Numéricas continuas: mediana global. Categóricas: moda. (Clase 4)
NUM_MED = ["m2", "lat", "lon", "n_dormitorios", "n_banos", "len_descripcion", "n_features",
    "floor", "pub_year", "pub_month"
]
CAT_MODE = ["property_type", "barrio"]

imp_med = SimpleImputer(strategy="median")
imp_med.fit(df_ent[NUM_MED])
df_ent[NUM_MED] = imp_med.transform(df_ent[NUM_MED])
df_ap[NUM_MED]  = imp_med.transform(df_ap[NUM_MED])

# Las flags f_* ya son 0/1 (no tienen NaN); por las dudas:
flag_cols = [c for c in df_ent.columns if c.startswith("f_")]
df_ent[flag_cols] = df_ent[flag_cols].fillna(0).astype(int)
df_ap[flag_cols]  = df_ap[flag_cols].fillna(0).astype(int)

# Categóricas con moda (sólo barrio queda con NaN; property_type ya está completo por filtro)
imp_mod = SimpleImputer(strategy="most_frequent")
imp_mod.fit(df_ent[CAT_MODE])
df_ent[CAT_MODE] = imp_mod.transform(df_ent[CAT_MODE])
df_ap[CAT_MODE]  = imp_mod.transform(df_ap[CAT_MODE])

EXPERIMENT_LOG["params"]["imputacion"] = {
    "numericas": {c: "median (global, fit on train)" for c in NUM_MED},
    "categoricas": {c: "most_frequent (fit on train)" for c in CAT_MODE},
    "marcador_m2_was_na": True,
}

print("\nFaltantes restantes por columna (top 10) tras imputación:")
print(df_ent.isna().sum().sort_values(ascending=False).head(10))

## 2.4. Hot Deck por descripción duplicada

Aplicación directa de la **imputación tipo Hot Deck** (Clase 4 — Datos Faltantes): "se reemplazan los faltantes con valores obtenidos de registros que son los más similares".

En este caso, el "faltante" es el `price` de las filas de `a_predecir`, y el "registro más similar" es una fila de `train` cuya **`description`** es exactamente la misma. Cuando aparece la misma descripción, casi siempre es la misma publicación (mismo aviso replicado). Construimos un diccionario `id_test → price_train` con la **mediana** de los precios encontrados (robusto si una descripción aparece varias veces con valores ruidosos), y al final lo aplicamos como override sobre la predicción del modelo.

> Esto **no** es un modelo: es exactamente la imputación Hot Deck. La construcción del diccionario también se hace **sólo con info del train** (no se usa el `price` del test — el test no lo tiene).

In [ ]:
# Hot Deck (Clase 4): override de la predicción del modelo cuando una fila del
# test tiene exactamente la misma descripción que una publicación del train.
# Es una imputación tipo donante: el "donante" es el precio mediano de las
# publicaciones del train con esa misma descripción.
#
# v4 — mejora respecto a v1/v2/v3:
# v1-v3 hacían `groupby("description")` con la cadena cruda. Eso falla cuando
# dos publicaciones difieren sólo en mayúsculas, tildes, espacios extra o
# puntuación. Normalizamos la descripción antes de armar el diccionario y
# antes de matchear el test, para subir el coverage del 6.9 % actual.
import unicodedata

def _norm_desc(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    s = str(s)
    # 1) saca acentos (NFKD + filtro de combining)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    # 2) lowercase
    s = s.lower()
    # 3) cualquier no alfanumérico → espacio (puntuación, símbolos, saltos)
    s = re.sub(r"[^0-9a-z\s]", " ", s)
    # 4) colapsar whitespace + strip
    s = re.sub(r"\s+", " ", s).strip()
    # 5) descripciones muy cortas (<20 chars) son demasiado genéricas para un
    #    Hot Deck por descripción exacta — las descartamos.
    return s if len(s) >= 20 else None

_eng = sqlite3.connect(f"{DIR}/entrenamiento.db")
_train_full = pd.read_sql(
    """
    SELECT id, description, price
    FROM entrenamiento
    WHERE operation_type = 'venta'
      AND currency_type  = 'dolares'
      AND price IS NOT NULL
      AND description    IS NOT NULL
    """,
    _eng,
)
_train_full["desc_norm"] = _train_full["description"].map(_norm_desc)

# Diccionario v1 (descripción cruda) — sólo para medir cuánto sumamos con la normalización
_price_by_desc_raw = (
    _train_full.dropna(subset=["description"])
    .groupby("description")["price"].median()
)
_test_desc_raw = df_ap["description"].dropna()
_hd_v1 = _test_desc_raw.map(_price_by_desc_raw).dropna().to_dict()

# Diccionario v4 (descripción normalizada)
price_by_desc = (
    _train_full.dropna(subset=["desc_norm"])
    .groupby("desc_norm")["price"].median()
)
test_desc_norm = df_ap["description"].map(_norm_desc).dropna()
hotdeck_dict = test_desc_norm.map(price_by_desc).dropna().to_dict()

cov_v1 = len(_hd_v1) / len(df_ap) * 100
cov_v4 = len(hotdeck_dict) / len(df_ap) * 100
ganancia = len(hotdeck_dict) - len(_hd_v1)
print(f"Hot Deck v1 (descripción cruda)        : {len(_hd_v1):>5} filas → {cov_v1:>5.2f}%")
print(f"Hot Deck v4 (descripción normalizada)  : {len(hotdeck_dict):>5} filas → {cov_v4:>5.2f}%")
print(f"Ganancia neta de coverage              : {ganancia:>+5} filas "
      f"({(cov_v4 - cov_v1):+.2f} pp)")

EXPERIMENT_LOG["params"]["hotdeck"] = {
    "key": "description_normalizada",
    "normalizacion": "lower + sin_acentos + sin_puntuacion + colapsar_espacios + min_20_chars",
    "agg": "median",
    "fuente": "train completo (venta + USD)",
}
EXPERIMENT_LOG["data"]["hotdeck_overrides"] = int(len(hotdeck_dict))
EXPERIMENT_LOG["data"]["hotdeck_coverage_pct"] = round(cov_v4, 2)
EXPERIMENT_LOG["data"]["hotdeck_overrides_baseline_v1"] = int(len(_hd_v1))
EXPERIMENT_LOG["data"]["hotdeck_coverage_baseline_v1_pct"] = round(cov_v1, 2)

del _train_full, price_by_desc, _price_by_desc_raw, _hd_v1, _test_desc_raw, test_desc_norm

In [ ]:
# --- Codificación de categóricas para el RandomForest ---
# El RF requiere features numéricos. Codificamos `barrio` y `property_type`
# con `pd.factorize` (numerización ordinal vista en Clase 2) usando el train
# como referencia y aplicando el mismo mapping al test (categorías nuevas → -1).
for col in ["barrio", "property_type"]:
    codes, uniques = pd.factorize(df_ent[col].astype(str))
    df_ent[f"{col}_id"] = codes
    mapping = {v: i for i, v in enumerate(uniques)}
    df_ap[f"{col}_id"] = df_ap[col].astype(str).map(mapping).fillna(-1).astype(int)

print("nuevos ids:", df_ent[["barrio_id", "property_type_id"]].nunique().to_dict())

In [ ]:
# --- 2.5 Features de contexto: DESACTIVADAS en v4 ----------------------------
# v2 había agregado `precio_mediano_barrio` y `precio_mediano_barrio_tipo` como
# target encoding de barrio. Mejoró holdout (-1.9K) pero EMPEORÓ Kaggle (+3.7K)
# → diagnóstico: target leakage. La mediana se calcula sobre TODO el train,
# entonces cada fila del train "ve" su propio precio agregado al promedio del
# barrio (especialmente en barrios chicos). Para entrenar honestamente habría
# que usarlo out-of-fold; eso es complejidad fuera del alcance de Clases 2-5.
# v4 vuelve al feature-set de v1 (sin target encoding).
print("v4: NO se agregan features de target encoding por barrio (decisión post-leakage v2).")
EXPERIMENT_LOG["params"]["target_encoding_barrio"] = "DESACTIVADO en v4 (leakage detectado en v2)"

## 3. Entrenamiento del modelos (AA)- ⛔⛔⛔ NO TOCAR NADA DE ESTA SECCIÓN ⛔⛔⛔

In [ ]:
# --- v6: snapshot de publication_date antes del filtro a numéricas ----------
# La celda siguiente reduce df_ent a columnas numéricas (requisito del RF) y
# tira `publication_date`. Lo guardamos alineado por índice para usarlo en
# el holdout temporal y el mini-ablation de las celdas 3.x.
_pub_date_serie = (
    df_ent["publication_date"].copy()
    if "publication_date" in df_ent.columns
    else None
)
print(f"[v6] snapshot publication_date guardado: "
      f"{None if _pub_date_serie is None else _pub_date_serie.notna().sum():,} "
      f"filas con fecha conocida.")

In [ ]:
# La creación de modelos requiere que todo el dataframe sea numérico
# Me quedo con las columnas numéricas solamente
df_ent = df_ent.select_dtypes('number')

X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

In [ ]:
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

# Definimos el valor de los hiperparámetros a usar por el modelo
n_estimators = 500
max_depth = 50

# Creamos el modelo a entrenar
reg = sk.ensemble.RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42)

# Entrenamos el modelo
_ = reg.fit(X_train, y_train)

# Cálculo del error en entrenamiento (train)
y_pred = reg.predict(X_train)
score_train = sk.metrics.root_mean_squared_error(y_train, y_pred)

# Cálculo del error en prueba (test)
y_pred = reg.predict(X_test)
score_test  = sk.metrics.root_mean_squared_error(y_test,  y_pred)

print(f"{n_estimators=} -- {max_depth=} --> {score_train=:.2f} - {score_test=:.2f}")

In [ ]:
# Captura del RMSE de holdout para el log del experimento
# (no toca el bloque de modelado: lee variables ya definidas).
EXPERIMENT_LOG["params"]["modelo"] = {
    "tipo": "RandomForestRegressor",
    "n_estimators": int(n_estimators),
    "max_depth": int(max_depth),
    "random_state": 42,
    "split": "train_test_split(test_size=0.2, random_state=42)",
}
EXPERIMENT_LOG["metricas"]["rmse_train"] = float(score_train)
EXPERIMENT_LOG["metricas"]["rmse_holdout"] = float(score_test)
EXPERIMENT_LOG["metricas"]["n_features"] = int(X.shape[1])
EXPERIMENT_LOG["metricas"]["features"]   = X.columns.tolist()

In [ ]:
# --- v6: CV5 multi-seed (3 seeds × 5 folds = 15 evaluaciones) -----------------
# Historia hasta acá:
#  · v3 mostró que el holdout único `random_state=42` tiene std ≈ 2K (mayor que
#    la diferencia entre v1, v2 y v3 ~1.8K) → el holdout suelto es ruido.
#  · v4 introdujo CV5 con un único seed como métrica primaria.
#  · v5 (8 features nuevas) MEJORÓ CV5 single-seed en ~5K pero EMPEORÓ Kaggle
#    en ~4K. Diagnóstico: multi-seed habría confirmado la mejora también
#    (no es varianza dentro del KFold), el problema fue distribution shift
#    train↔test público.
#
# v6 endurece la validación en dos direcciones complementarias:
#   1) Multi-seed (acá) → estabiliza la varianza intra-KFold y fuerza a que la
#      mejora sea robusta a la partición aleatoria (ya no a 1 sola).
#   2) Holdout TEMPORAL (celda siguiente) → captura el distribution shift
#      que multi-seed no ve (porque KFold reparte años uniformemente).
# (Este bloque NO toca el modelo `reg` entrenado arriba — instancia un RF
#  nuevo por fold; el holdout único de cell 28 se conserva por compat.)
import time
import numpy as _np

SEEDS_CV5 = [42, 7, 2024]
print(f"[CV5] arrancando multi-seed: {len(SEEDS_CV5)} seeds × 5 folds = "
      f"{len(SEEDS_CV5)*5} evaluaciones")
print(f"[CV5] n_est={n_estimators}, max_depth={max_depth}")
print(f"[CV5] esto agrega ~30-45 min al runtime (vs ~10-15 con un solo seed).")
_t0 = time.time()
_cv_rmses_all = []
_cv_per_seed = {}
for _seed in SEEDS_CV5:
    _kf = sk.model_selection.KFold(n_splits=5, shuffle=True, random_state=_seed)
    _seed_rmses = []
    for _fold, (_tr, _te) in enumerate(_kf.split(X, y)):
        _reg_cv = sk.ensemble.RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            n_jobs=-1, random_state=42,
        )
        _reg_cv.fit(X.iloc[_tr], y.iloc[_tr])
        _pred = _reg_cv.predict(X.iloc[_te])
        _rmse = float(sk.metrics.root_mean_squared_error(y.iloc[_te], _pred))
        _seed_rmses.append(_rmse)
        _cv_rmses_all.append(_rmse)
        print(f"[CV5] seed={_seed} fold {_fold}: RMSE = {_rmse:,.2f}   "
              f"(n_train={len(_tr):,}, n_test={len(_te):,})")
    _cv_per_seed[str(_seed)] = {
        "mean":  float(_np.mean(_seed_rmses)),
        "std":   float(_np.std(_seed_rmses)),
        "folds": _seed_rmses,
    }
    print(f"[CV5] seed={_seed} → mean={_cv_per_seed[str(_seed)]['mean']:,.2f} "
          f"std={_cv_per_seed[str(_seed)]['std']:,.2f}")

_cv_mean = float(_np.mean(_cv_rmses_all))
_cv_std  = float(_np.std(_cv_rmses_all))
print(f"\n[CV5] AGREGADO ({len(_cv_rmses_all)} evals): "
      f"media={_cv_mean:,.2f}   std={_cv_std:,.2f}   "
      f"min={min(_cv_rmses_all):,.2f}   max={max(_cv_rmses_all):,.2f}")
print(f"[CV5] tiempo total: {time.time()-_t0:.1f}s")

EXPERIMENT_LOG["metricas"]["rmse_cv5_folds"]    = _cv_rmses_all
EXPERIMENT_LOG["metricas"]["rmse_cv5_mean"]     = _cv_mean
EXPERIMENT_LOG["metricas"]["rmse_cv5_std"]      = _cv_std
EXPERIMENT_LOG["metricas"]["rmse_cv5_per_seed"] = _cv_per_seed
EXPERIMENT_LOG["params"]["validacion_primaria"] = (
    f"KFold(5) multi-seed seeds={SEEDS_CV5}"
)

del _reg_cv, _kf, _seed_rmses

In [ ]:
# --- v6: Holdout TEMPORAL (último 20% por publication_date) -----------------
# Motivación (lección de v5):
#   v5 mejoró CV5 (multi-seed o no) y empeoró Kaggle. KFold aleatorio reparte
#   los años uniformemente entre folds, así que NO ve el shift temporal entre
#   train y test público. Acá ordenamos por fecha y dejamos como validation
#   las publicaciones más recientes — un proxy razonable de "el test es de
#   un período distinto".
#
# Reglas:
#  · Las filas SIN publication_date se EXCLUYEN del holdout (no del train final
#    de Kaggle). Acá sólo medimos sobre el subset con fecha conocida.
#  · Es validación auxiliar: no toca `reg`, no cambia el modelo final que
#    se entrena en X, y para Kaggle.
#  · Split = un único corte ordenado (no es KFold porque no queremos shuffle).
#    Sigue siendo "holdout sin shuffle", herramienta de Clase 3.
import time as _time_h

# `publication_date` viene en dos formatos:
#   "1 ene 2026", "30 nov 2025", ...   ← absolutos en español
#   "Hace 1 semana, 17 horas", ...     ← relativos al scrape (no parseables)
# pd.to_datetime() falla en ambos por idioma + offset mixto. Usamos el mismo
# regex que `extract_temporal_features` y ordenamos por (año, mes) numérico.
# Las filas con formato relativo o no-parseable se EXCLUYEN del holdout
# (no del train final de Kaggle).
def _pub_score(s, _meses={"ene":1,"feb":2,"mar":3,"abr":4,"may":5,"jun":6,
                          "jul":7,"ago":8,"sept":9,"sep":9,"oct":10,
                          "nov":11,"dic":12},
               _pat=re.compile(r"(\d{1,2})\s+(\w+)\s+(\d{4})")):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    m = _pat.match(str(s).strip())
    if not m:
        return None
    _, mes, a = m.groups()
    mn = _meses.get(mes.lower())
    if mn is None:
        return None
    return int(a) * 100 + int(mn)  # YYYYMM, ordenable

if _pub_date_serie is None:
    print("[holdout-temporal] publication_date no disponible; salteo.")
    rmse_temporal      = None
    n_train_temporal   = None
    n_val_temporal     = None
    cut_date_temporal  = None
else:
    _pub_aligned = _pub_date_serie.reindex(X.index)
    _scores = _pub_aligned.map(_pub_score)
    _scores = _scores.dropna()
    _scores = _scores.astype(int)
    print(f"[holdout-temporal] filas parseadas: {len(_scores):,} / {len(_pub_aligned):,}")

    if len(_scores) < 1000:
        print(f"[holdout-temporal] muy pocas filas con fecha ({len(_scores)}); salteo.")
        rmse_temporal     = None
        n_train_temporal  = None
        n_val_temporal    = None
        cut_date_temporal = None
    else:
        _orden = _scores.sort_values().index
        _n     = len(_orden)
        _cut   = int(_n * 0.80)
        _idx_tr = _orden[:_cut]
        _idx_vl = _orden[_cut:]

        _Xtr_t = X.loc[_idx_tr]
        _ytr_t = y.loc[_idx_tr]
        _Xvl_t = X.loc[_idx_vl]
        _yvl_t = y.loc[_idx_vl]

        _cut_yyyymm = int(_scores.loc[_idx_vl[0]])
        _cut_human  = f"{_cut_yyyymm // 100:04d}-{_cut_yyyymm % 100:02d}"
        print(f"[holdout-temporal] n_train={len(_Xtr_t):,}  n_val={len(_Xvl_t):,}  "
              f"corte = {_cut_human}")
        _t0 = _time_h.time()
        _reg_t = sk.ensemble.RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            n_jobs=-1, random_state=42,
        )
        _reg_t.fit(_Xtr_t, _ytr_t)
        _pred_t = _reg_t.predict(_Xvl_t)
        rmse_temporal = float(sk.metrics.root_mean_squared_error(_yvl_t, _pred_t))
        print(f"[holdout-temporal] RMSE = {rmse_temporal:,.2f}   "
              f"tiempo: {_time_h.time()-_t0:.1f}s")

        n_train_temporal  = int(len(_Xtr_t))
        n_val_temporal    = int(len(_Xvl_t))
        cut_date_temporal = _cut_human
        del _reg_t, _Xtr_t, _ytr_t, _Xvl_t, _yvl_t, _pred_t

EXPERIMENT_LOG["metricas"]["rmse_holdout_temporal"]      = rmse_temporal
EXPERIMENT_LOG["metricas"]["holdout_temporal_n_train"]   = n_train_temporal
EXPERIMENT_LOG["metricas"]["holdout_temporal_n_val"]     = n_val_temporal
EXPERIMENT_LOG["metricas"]["holdout_temporal_cut_date"]  = cut_date_temporal
EXPERIMENT_LOG["params"]["validacion_secundaria"]        = (
    "Holdout temporal último 20% ordenado por publication_date"
)

In [ ]:
# --- v6: Mini-ablation de las 8 features de v5 (LOCAL, sin Kaggle) ----------
# Objetivo: aislar qué bloque de las 8 features de v5 es responsable del shift
# que vimos en Kaggle (97 498 vs 93 151 de v4). Probamos 4 sub-conjuntos:
#
#   v6.A = v4 baseline                                     (sin features v5)
#   v6.B = v4 + sólo `floor`
#   v6.C = v4 + sólo temporales (pub_year, pub_month)
#   v6.D = v4 + sólo booleanos (is_a_estrenar, is_reciclado, has_suite,
#                               has_service_room, is_near_subway)
#
# Para cada uno medimos CV5 (single-seed, 5 evals) + holdout temporal, y
# escribimos la tabla a entregas/entrega_2/ablation_v5.md. Sin submits a Kaggle:
# el output sirve para diseñar Entrega 3.
import time as _time_a
from pathlib import Path as _PathA

V5_FEATURES_ALL = [
    "floor", "is_a_estrenar", "is_reciclado", "has_suite",
    "has_service_room", "is_near_subway", "pub_year", "pub_month",
]
SUBSETS = {
    "v6.A_baseline_v4":    [],
    "v6.B_solo_floor":     ["floor"],
    "v6.C_solo_temporales":["pub_year", "pub_month"],
    "v6.D_solo_booleanos": ["is_a_estrenar", "is_reciclado", "has_suite",
                            "has_service_room", "is_near_subway"],
}

# Snapshot del conjunto temporal (mismo split YYYYMM que la celda anterior).
# Reusamos `_pub_score` definido en la celda 3.x del holdout temporal.
_have_temporal = (_pub_date_serie is not None and rmse_temporal is not None)
if _have_temporal:
    _pub_aligned_a = _pub_date_serie.reindex(X.index)
    _scores_a = _pub_aligned_a.map(_pub_score).dropna().astype(int)
    _orden_a = _scores_a.sort_values().index
    _cut_a   = int(len(_orden_a) * 0.80)
    _idx_tr_a = _orden_a[:_cut_a]
    _idx_vl_a = _orden_a[_cut_a:]

ablation_rows = []
for _name, _extra in SUBSETS.items():
    _to_drop = [c for c in V5_FEATURES_ALL if c not in _extra and c in X.columns]
    _X_sub = X.drop(columns=_to_drop)
    print(f"\n[ablation] {_name}: features={_X_sub.shape[1]}  "
          f"(drop {len(_to_drop)}: {_to_drop})")

    # CV5 single-seed
    _t0 = _time_a.time()
    _kf_a = sk.model_selection.KFold(n_splits=5, shuffle=True, random_state=42)
    _r = []
    for _fold, (_tr, _te) in enumerate(_kf_a.split(_X_sub, y)):
        _reg_a = sk.ensemble.RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            n_jobs=-1, random_state=42,
        )
        _reg_a.fit(_X_sub.iloc[_tr], y.iloc[_tr])
        _r.append(float(sk.metrics.root_mean_squared_error(
            y.iloc[_te], _reg_a.predict(_X_sub.iloc[_te]))))
    _cv_m = float(_np.mean(_r))
    _cv_s = float(_np.std(_r))
    print(f"[ablation] {_name}: CV5 mean={_cv_m:,.2f}  std={_cv_s:,.2f}  "
          f"tiempo={_time_a.time()-_t0:.1f}s")

    # Holdout temporal con el mismo split de la celda anterior
    if _have_temporal:
        _reg_a2 = sk.ensemble.RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            n_jobs=-1, random_state=42,
        )
        _reg_a2.fit(_X_sub.loc[_idx_tr_a], y.loc[_idx_tr_a])
        _rmse_temp_a = float(sk.metrics.root_mean_squared_error(
            y.loc[_idx_vl_a], _reg_a2.predict(_X_sub.loc[_idx_vl_a])))
        print(f"[ablation] {_name}: holdout_temporal RMSE={_rmse_temp_a:,.2f}")
        del _reg_a2
    else:
        _rmse_temp_a = None

    ablation_rows.append({
        "subset":            _name,
        "extra_features":    _extra,
        "n_features_total":  int(_X_sub.shape[1]),
        "cv5_mean":          _cv_m,
        "cv5_std":           _cv_s,
        "holdout_temporal":  _rmse_temp_a,
    })

# Baseline v6.A → deltas
_base = next((r for r in ablation_rows if r["subset"].startswith("v6.A")), None)
for r in ablation_rows:
    r["delta_cv5_vs_v4"]      = (r["cv5_mean"] - _base["cv5_mean"]) if _base else None
    r["delta_temporal_vs_v4"] = ((r["holdout_temporal"] - _base["holdout_temporal"])
                                 if _base and r["holdout_temporal"] is not None
                                    and _base["holdout_temporal"] is not None else None)

# Persistir tabla en entregas/entrega_2/ablation_v5.md
try:
    _ablation_dir = _PathA("entregas/entrega_2")
    _ablation_dir.mkdir(parents=True, exist_ok=True)
    _md_path = _ablation_dir / "ablation_v5.md"
    _lines = [
        "# Mini-ablation de las 8 features de v5",
        "",
        ("Objetivo: identificar qué bloque de las features que v5 introdujo es "
         "responsable del distribution shift que se vio en Kaggle "
         "(97 498 vs los 93 151 de v4). Sin submits a Kaggle; sólo CV5 + "
         "holdout temporal local."),
        "",
        ("Baseline v6.A = v4 (sin features de v5). Deltas se computan vs v6.A. "
         "**Negativo = mejora; positivo = empeora.**"),
        "",
        "| Sub-conjunto | Features extra | n_features | CV5 mean ± std | "
        "Holdout temporal | Δ CV5 vs v4 | Δ holdout temporal vs v4 |",
        "|---|---|---:|---:|---:|---:|---:|",
    ]
    for r in ablation_rows:
        _extra_s = ", ".join(r["extra_features"]) if r["extra_features"] else "—"
        _ht = (f"{r['holdout_temporal']:,.0f}" if r["holdout_temporal"] is not None else "—")
        _dcv = (f"{r['delta_cv5_vs_v4']:+,.0f}" if r["delta_cv5_vs_v4"] is not None else "—")
        _dht = (f"{r['delta_temporal_vs_v4']:+,.0f}"
                if r["delta_temporal_vs_v4"] is not None else "—")
        _lines.append(
            f"| {r['subset']} | {_extra_s} | {r['n_features_total']} | "
            f"{r['cv5_mean']:,.0f} ± {r['cv5_std']:,.0f} | {_ht} | {_dcv} | {_dht} |"
        )
    _lines.append("")
    _lines.append(f"_Generado automáticamente por la celda de mini-ablation de v6._")
    _md_path.write_text("\n".join(_lines), encoding="utf-8")
    print(f"\n[ablation] tabla escrita en {_md_path}")
except Exception as _e:
    print(f"[ablation] no pude escribir el .md: {_e}")

EXPERIMENT_LOG["metricas"]["ablation_v5"] = ablation_rows
del _kf_a, _reg_a

## 3.1. (Opcional) Validación Cruzada para mejorar el modelo a partir de los datos

Esta técnica permite mejorar el modelo adaptando los hiperparámetros a los datos seleccionados

**NOTA**: Esta técnica puede tardar mucho, se recomienda ir probando pocos valores en paralelo al resto de los analisis.

In [ ]:
# definimos el valor de los hiperparámetros
n_estimators = 500
max_depth = 50

# Creamos el modelo
# No cambiar RandomForestRegressor
reg = sk.ensemble.RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42)

scores_train = []
scores_test = []

# Validación cruzada, 10 folds, shuffle antes, semilla aleatoria
kf = sk.model_selection.KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    # Partimos el fold en entrenamiento y prueba...
    X_train, X_test, y_train, y_test = X.iloc[train_index], X.iloc[test_index], y.iloc[train_index], y.iloc[test_index]

    # Entrenamos el modelo en entramiento
    reg.fit(X_train, y_train)

    # Predecimos en train
    y_pred = reg.predict(X_train)

    # Medimos la performance de la predicción en entramiento
    score_train = sk.metrics.root_mean_squared_error(y_train, y_pred)

    # Predecimos en test
    y_pred = reg.predict(X_test)

    # Medimos la performance de la predicción en prueba
    score_test = sk.metrics.root_mean_squared_error(y_test, y_pred)

    print("\t", f"{fold=}, {score_train=} {score_test=}")


## 3.2. Análisis de la importancia de variables en el modelo (opcional)

Una manera visual de entender a que variable el modelo le está prestando mayor atención.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feat_importances = pd.Series(reg.feature_importances_, index=X.columns)

# gráfico de barras horizontales
feat_importances.nlargest(10).plot(kind='barh');

## 4. Solución para subir Kaggle

`df_ap` ya viene preprocesado en paralelo a `df_ent` (mismas imputaciones, mismos atributos nuevos), por lo que sólo tenemos que entrenar el modelo final, predecir y aplicar el override Hot Deck.

In [ ]:
# df_ap ya fue cargado en la sección 0 y preprocesado junto con df_ent.
df_ap.head(2)

In [ ]:
df_ap.shape

In [ ]:
# `X` e `y` ya quedaron definidos en la sección 3. Si la celda de CV no se
# ejecutó, `mejor_combinacion` no existe → usamos los hiperparámetros base
# de la sección 3 (los mismos que reportan el RMSE comparable a la entrega 1).
try:
    mejor_combinacion  # type: ignore[name-defined]
except NameError:
    mejor_combinacion = {"n_estimators": 500, "max_depth": 50}

best_n_estimators = mejor_combinacion.get("n_estimators") or 500
best_max_depth    = mejor_combinacion.get("max_depth")    or 50

reg = sk.ensemble.RandomForestRegressor(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    n_jobs=-1,
    random_state=42,
)

reg.fit(X, y)
print(f"Modelo final entrenado con {len(X)} filas y {X.shape[1]} features.")

## 4.2. Generación del archivo para Kaggle

In [ ]:
# df_ap ya tiene aplicadas las mismas transformaciones que df_ent
# (parseo de features, imputación, factorización de barrio/property_type, etc.).
# Sólo nos quedamos con las columnas numéricas que vio el modelo.
df_ap_num = df_ap.select_dtypes("number")

# Si quedaran NaN residuales (algún borde no cubierto), los rellenamos con 0
# como red de seguridad (no debería haber).
df_ap_num = df_ap_num.fillna(0)

X_ap = df_ap_num.reindex(columns=X.columns, fill_value=0)

y_pred_ap = reg.predict(X_ap)
print(f"Predicciones generadas para {len(y_pred_ap)} filas.")
print(pd.Series(y_pred_ap).describe(percentiles=[.05, .5, .95]))

In [ ]:
# 1) Predicción cruda del modelo
df_ap["price"] = y_pred_ap

# 2) Override Hot Deck: para cada id de test que matcheó por descripción
#    con una publicación de venta del train, reemplazamos la predicción por
#    la mediana del precio observado.
n_override = int(df_ap.index.isin(hotdeck_dict).sum())
df_ap.loc[df_ap.index.isin(hotdeck_dict), "price"] = (
    df_ap.loc[df_ap.index.isin(hotdeck_dict)].index.map(hotdeck_dict)
)
print(f"Hot Deck aplicado a {n_override} filas ({n_override/len(df_ap)*100:.1f}%).")

# 3) Sanity check: nada negativo, nada NaN
df_ap["price"] = df_ap["price"].clip(lower=1).fillna(df_ap["price"].median())

# 4) Salida: CSV (formato Kaggle) + JSON de metadatos + actualización de leaderboard.
#    El CSV es lo que se sube a Kaggle. El JSON queda como registro auditable
#    del experimento y permite reproducir / comparar corridas.
out_dir = DIR if IN_COLAB else f"entregas/{ENTREGA}"
os.makedirs(out_dir, exist_ok=True)

stem      = f"solucion-{ENTREGA.replace('_', '')}-{NOMBRE}"
csv_path  = f"{out_dir}/{stem}.csv"
json_path = f"{out_dir}/{stem}.json"

df_ap["price"].to_csv(csv_path)

# Hash del CSV: detecta si dos corridas generan exactamente la misma submission.
with open(csv_path, "rb") as _f:
    csv_md5 = hashlib.md5(_f.read()).hexdigest()

EXPERIMENT_LOG["data"]["test_predicciones"] = int(len(df_ap))
EXPERIMENT_LOG["data"]["hotdeck_overrides_aplicados"] = n_override
EXPERIMENT_LOG["metricas"]["pred_p05"]     = float(df_ap["price"].quantile(0.05))
EXPERIMENT_LOG["metricas"]["pred_mediana"] = float(df_ap["price"].median())
EXPERIMENT_LOG["metricas"]["pred_p95"]     = float(df_ap["price"].quantile(0.95))
EXPERIMENT_LOG["output"] = {
    "csv": csv_path,
    "csv_md5": csv_md5,
    "json": json_path,
    "kaggle_submit_message": f"{ENTREGA}/{NOMBRE}: {DESCRIPCION}",
}

with open(json_path, "w", encoding="utf-8") as _f:
    json.dump(EXPERIMENT_LOG, _f, indent=2, ensure_ascii=False)

# 5) Actualización del leaderboard local (sólo cuando corremos fuera de Colab,
#    porque en Drive no tiene sentido versionar este markdown).
if not IN_COLAB:
    lb_path = f"entregas/{ENTREGA}/leaderboard.md"
    header = (
        "# Leaderboard local — " + ENTREGA + "\n\n"
        "| nombre | fecha | RMSE CV5 (mean ± std) | RMSE holdout temporal | RMSE holdout | RMSE Kaggle | hotdeck % | descripción | csv md5 |\n"
        "|---|---|---:|---:|---:|---:|---:|---|---|\n"
    )
    if not os.path.exists(lb_path):
        with open(lb_path, "w", encoding="utf-8") as _f:
            _f.write(header)
    cv_mean = EXPERIMENT_LOG["metricas"].get("rmse_cv5_mean")
    cv_std  = EXPERIMENT_LOG["metricas"].get("rmse_cv5_std")
    cv_cell = f"{cv_mean:,.2f} ± {cv_std:,.0f}" if cv_mean is not None else "—"
    rmse_temp = EXPERIMENT_LOG["metricas"].get("rmse_holdout_temporal")
    temp_cell = f"{rmse_temp:.2f}" if rmse_temp is not None else "—"
    fila = (
        f"| {NOMBRE} "
        f"| {EXPERIMENT_LOG['timestamp'][:16].replace('T', ' ')} "
        f"| {cv_cell} "
        f"| {temp_cell} "
        f"| {EXPERIMENT_LOG['metricas']['rmse_holdout']:.2f} "
        f"| _pendiente_ "
        f"| {EXPERIMENT_LOG['data']['hotdeck_coverage_pct']:.1f}% "
        f"| {DESCRIPCION} "
        f"| `{csv_md5[:8]}` |\n"
    )
    with open(lb_path, "a", encoding="utf-8") as _f:
        _f.write(fila)
    print(f"Leaderboard actualizado: {lb_path}")

print("---")
print(f"CSV : {csv_path}")
print(f"JSON: {json_path}")
print(f"MD5 : {csv_md5}")
print(f"Para Kaggle (Submission Description): {EXPERIMENT_LOG['output']['kaggle_submit_message']}")
df_ap["price"].describe(percentiles=[.05, .5, .95])